# Teste do Modelo - Previsão de Tickets totais

In [39]:
import os
import pickle
import sys
import warnings

warnings.filterwarnings("ignore")
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
sys.path.insert(0, project_root)

In [40]:
model_name = "SARIMAX"
model_path = f"../../../models/{model_name}_model.pkl"

with open(model_path, "rb") as f:
    model = pickle.load(f)

In [41]:
steps = 30  # número de dias para prever
if model_name in ["AR", "MA", "ARMA", "ARIMA", "SARIMA", "SARIMAX", "Holt-Winters"]:
    predictions = model.forecast(steps=steps)

elif model_name == "Auto-ARIMA":
    predictions = model.predict(n_periods=steps)

elif model_name == "Prophet":
    # Prophet precisa de um DataFrame com datas
    future = model.make_future_dataframe(periods=steps)
    forecast = model.predict(future)
    predictions = forecast["yhat"].tail(steps).values

elif model_name == "LightGBM":
    import numpy as np
    import pandas as pd

    # O modelo foi treinado com índices numéricos simples
    # Precisamos continuar a sequência de onde parou o treino

    # Carregar dados de treino para saber o último índice
    from src.utils.extract_data import get_data

    df = get_data("../../sql/all_dates2.sql")
    train_size = int(len(df) * 0.8)

    # Criar índices para os próximos 'steps' dias
    last_index = train_size - 1
    future_indices = np.arange(last_index + 1, last_index + 1 + steps).reshape(-1, 1)

    # Prever usando o modelo carregado
    predictions = model.predict(future_indices)
    predictions = np.maximum(
        predictions, 0
    )  # Garantir que não haja previsões negativas

else:
    raise ValueError(f"Modelo desconhecido: {model_name}")

In [42]:
future_df = pd.DataFrame(
    {"Date": pd.date_range(start="2025-10-01", periods=10, freq="D")}
)
print(future_df)

        Date
0 2025-10-01
1 2025-10-02
2 2025-10-03
3 2025-10-04
4 2025-10-05
5 2025-10-06
6 2025-10-07
7 2025-10-08
8 2025-10-09
9 2025-10-10


In [43]:
print(f"Previsões para os próximos {steps} dias:")
print(predictions)

Previsões para os próximos 30 dias:
2024-08-30    47.154583
2024-08-31    40.315422
2024-09-01    45.434763
2024-09-02    46.712003
2024-09-03    46.102851
2024-09-04    42.768340
2024-09-05    54.610798
2024-09-06    48.419860
2024-09-07    46.241252
2024-09-08    42.384108
2024-09-09    45.253611
2024-09-10    46.909115
2024-09-11    46.343906
2024-09-12    48.490994
2024-09-13    45.483748
2024-09-14    44.936963
2024-09-15    47.432650
2024-09-16    48.598601
2024-09-17    43.526984
2024-09-18    45.106334
2024-09-19    47.810897
2024-09-20    48.911410
2024-09-21    47.917388
2024-09-22    47.228070
2024-09-23    46.330640
2024-09-24    45.860129
2024-09-25    45.878527
2024-09-26    45.871588
2024-09-27    47.065597
2024-09-28    46.500094
Freq: D, Name: predicted_mean, dtype: float64


In [44]:
predictions

2024-08-30    47.154583
2024-08-31    40.315422
2024-09-01    45.434763
2024-09-02    46.712003
2024-09-03    46.102851
2024-09-04    42.768340
2024-09-05    54.610798
2024-09-06    48.419860
2024-09-07    46.241252
2024-09-08    42.384108
2024-09-09    45.253611
2024-09-10    46.909115
2024-09-11    46.343906
2024-09-12    48.490994
2024-09-13    45.483748
2024-09-14    44.936963
2024-09-15    47.432650
2024-09-16    48.598601
2024-09-17    43.526984
2024-09-18    45.106334
2024-09-19    47.810897
2024-09-20    48.911410
2024-09-21    47.917388
2024-09-22    47.228070
2024-09-23    46.330640
2024-09-24    45.860129
2024-09-25    45.878527
2024-09-26    45.871588
2024-09-27    47.065597
2024-09-28    46.500094
Freq: D, Name: predicted_mean, dtype: float64